# Feature EDA

Merges CGI datasets from all three species, extracts 3-mer composition
features from each CGI sequence, and explores species-level separability
via 3D scatter and PCA.

**Input**: `dataset/processed/{species}_islands.csv`

**Output**: `dataset/processed/cpg_islands.csv` (merged, labeled),
            `dataset/processed/3mer_features.csv`

In [ ]:
import pandas as pd
import plotly.express as px
from itertools import product
from collections import Counter
from sklearn.decomposition import PCA

In [ ]:
arab_island_csv = pd.read_csv('../dataset/processed/arabidopsis_islands.csv')
must_island_csv = pd.read_csv('../dataset/processed/mustard_islands.csv')
cabb_island_csv = pd.read_csv('../dataset/processed/c_cabbage_islands.csv')

In [ ]:
arab_island_csv.head()

,id,chrom,start,end,seq,gc-content,length,obs/exp
0,AT1G14150,Chr1,479,807,TTCTGATAGGCGTGTAGAACAGGTGTCTGATAATCGCCGGAGATAG...,0.500000,328,0.987506
1,AT1G14140,Chr1,289,507,CGACAAGCTCGTTTGCCGCCTGAACGTAATCATTCCGGCGACTCTC...,0.509174,218,1.159574
2,AT1G64960,Chr1,0,211,GCCGTTCGGAGCGGGTAGAAGGATTTGTCCAGGGATTGGGCTAGCG...,0.516588,211,0.703855
3,AT1G08270,Chr1,0,213,AGGAGAAGCTGCTCCGACGAATCGCCACTCTGCAGATGAGTGTAGC...,0.502347,213,1.482418
4,AT1G08450,Chr1,80,515,TATAAAAGTTCCTTCTTTTCTGCGAACCTTAGAGCGATGTACCTGA...,0.505747,435,1.413750
...,...,...,...,...,...,...,...,...
1173,ATMG01220,ChrM,176,376,GCCGTCGTTTATGATAGGTCGGTCTAGTGAGAGCAAATGATCAAGC...,0.500000,200,0.644122
1174,ATMG00860,ChrM,0,327,AGCACATCGGTGTGGTCTGCAAATCACAGAAACTCGTGGATTTGGG...,0.504587,327,0.704308
1175,ATMG00520,ChrM,155,936,GGAAGGCTAACTGTGCTTGCTGGCTGGGAGCTGTATGAGCGGTAAC...,0.535211,781,0.766742
1176,ATMG00650,ChrM,473,1000,GTTTTTTTGTCTTTGTTGTTAGTCACCAACTTATTTACCCAGAAAT...,0.516129,527,0.826287


In [ ]:
must_island_csv.head()

,id,chrom,start,end,seq,gc-content,length,obs/exp
0,BjuVA01G00020.1,AA_Chr01,0,386,GATCTGGTGGTTCCGGCGGGGAGAGAGTTTCGAATGGCACCTGAAA...,0.546632,386,1.079936
1,BjuVA01G00040.1,AA_Chr01,31,600,TTAAAAAAAAAAAAGAGAGAGAGTGTGTTTTAACACTTTTACCTTC...,0.509666,569,0.963753
2,BjuVA01G00050.1,AA_Chr01,61,261,CGAGTCCCACATTGCTTATCAAGACCTGAAAGAAGTGGGAGCTCCC...,0.520000,200,0.614439
3,BjuVA01G00060.1,AA_Chr01,0,323,GGGCATCTCGTTGAAAGCGAATATTGCAGCTGCTGTTGCTCGCCAG...,0.507740,323,0.769048
4,BjuVA01G00070.1,AA_Chr01,0,209,GACGTTGGCGAAGGTCTTGAACTGGCTAGAAAGAGGGGGTGACTTG...,0.507177,209,0.736780
...,...,...,...,...,...,...,...,...
12096,BjuV_Contig01063G00470.1,Contig01063,485,1000,TCTTTGTTGTTAGTCACCAAGTTATTTACCCAGAAATAGATAAGGG...,0.524272,515,0.791395
12097,BjuV_Contig01063G00480.1,Contig01063,64,1000,TCAGTAGGTTACTTATGCTTCGACGATTTGACTCGTGCCATTTCTC...,0.536325,936,0.956918
12098,BjuV_Contig01063G00510.1,Contig01063,744,1000,GAAGCACGACCGAGGTCTTCGTAGTCCAAATAATCAAACCATCAAT...,0.554688,256,0.914830
12099,BjuV_Contig01063G00520.1,Contig01063,0,325,ATTTGATGGAATGGCGTGGCGATAGACTTCGCTTCTACGACTGCCT...,0.520000,325,1.093356


In [ ]:
cabb_island_csv.head()

,id,chrom,start,end,seq,gc-content,length,obs/exp
0,BraA01g000080.4C.1,A01,264,473,CCATGCTCTCGTCTCCTCCTTGGTGAGGATGGGTGATGTTGAAGCT...,0.502392,209,0.613353
1,BraA01g000080.4C.1,A01,378,627,TGAATGTCTACGTTAAAAACGACCAACTAGACAAGGCTGAAGGGTT...,0.506024,249,0.659952
2,BraA01g000140.4C.1,A01,0,695,TTTCCTTACTTACCTTCGTCGAGCACGATTCCGACGTTCAAGGACT...,0.523741,695,0.895536
3,BraA01g000140.4C.2,A01,0,658,TCAAGGACTTGAAGAGGTGGGATTCAACGAACTTCTCGGCTCCATC...,0.525836,658,0.832257
4,BraA01g000170.4C.1,A01,0,263,CCGGCTTCCCAGTTGGCAGGAAGAACCTTCTTAGACCAGAGCATTT...,0.509506,263,0.703052
...,...,...,...,...,...,...,...,...
6066,BraA10g033540.4C.1,A10,1,208,GGACATGGGGGTGCTTGTTTCGAGGAAACAGCACGGGGAACATCAC...,0.502415,207,0.676471
6067,BraA10g033560.4C.1,A10,194,607,CTGTCCACTCAGAGTTTGGCTCGCTCCATGACATCGGTCTCACGCC...,0.501211,413,0.658878
6068,BraA10g033750.4C.1,A10,74,753,CGCTTGTACTCCTCAATGTTAACCGAAATGGGCTTTGTTAATTAAG...,0.530191,679,1.253512
6069,BraA10g033840.4C.1,A10,0,324,CGGAAAAGGCTCCGGCAACCCAATCTGATGATGAAGCCCTAACGTC...,0.509259,324,1.005914


In [ ]:
# Check for missing values
print(f'Arabidopsis missing value: {arab_island_csv.isnull().sum()}')
print(f'Mustard missing value: {must_island_csv.isnull().sum()}')
print(f'Chinese cabbage missing value: {cabb_island_csv.isnull().sum()}')

Arabidopsis missing value: id            0
chrom         0
start         0
end           0
seq           0
gc-content    0
length        0
obs/exp       0
dtype: int64
Mustard missing value: id            0
chrom         0
start         0
end           0
seq           0
gc-content    0
length        0
obs/exp       0
dtype: int64
Chinese cabbage missing value: id            0
chrom         0
start         0
end           0
seq           0
gc-content    0
length        0
obs/exp       0
dtype: int64


In [ ]:
# Concatenate all three species into one DataFrame, tagging each row
# with its species of origin (used later for grouping/coloring in plots
# and as the classification target in decision_tree.ipynb)

arab_island_csv['species'] = 'arabidopsis'
must_island_csv['species'] = 'mustard'
cabb_island_csv['species'] = 'chinese_cabbage'

islands = pd.concat([arab_island_csv, must_island_csv, cabb_island_csv], axis=0, ignore_index=True)

In [ ]:
islands

,id,chrom,start,end,seq,gc-content,length,obs/exp,species
0,AT1G14150,Chr1,479,807,TTCTGATAGGCGTGTAGAACAGGTGTCTGATAATCGCCGGAGATAG...,0.500000,328,0.987506,arabidopsis
1,AT1G14140,Chr1,289,507,CGACAAGCTCGTTTGCCGCCTGAACGTAATCATTCCGGCGACTCTC...,0.509174,218,1.159574,arabidopsis
2,AT1G64960,Chr1,0,211,GCCGTTCGGAGCGGGTAGAAGGATTTGTCCAGGGATTGGGCTAGCG...,0.516588,211,0.703855,arabidopsis
3,AT1G08270,Chr1,0,213,AGGAGAAGCTGCTCCGACGAATCGCCACTCTGCAGATGAGTGTAGC...,0.502347,213,1.482418,arabidopsis
4,AT1G08450,Chr1,80,515,TATAAAAGTTCCTTCTTTTCTGCGAACCTTAGAGCGATGTACCTGA...,0.505747,435,1.413750,arabidopsis
...,...,...,...,...,...,...,...,...,...
19345,BraA10g033540.4C.1,A10,1,208,GGACATGGGGGTGCTTGTTTCGAGGAAACAGCACGGGGAACATCAC...,0.502415,207,0.676471,chinese_cabbage
19346,BraA10g033560.4C.1,A10,194,607,CTGTCCACTCAGAGTTTGGCTCGCTCCATGACATCGGTCTCACGCC...,0.501211,413,0.658878,chinese_cabbage
19347,BraA10g033750.4C.1,A10,74,753,CGCTTGTACTCCTCAATGTTAACCGAAATGGGCTTTGTTAATTAAG...,0.530191,679,1.253512,chinese_cabbage
19348,BraA10g033840.4C.1,A10,0,324,CGGAAAAGGCTCCGGCAACCCAATCTGATGATGAAGCCCTAACGTC...,0.509259,324,1.005914,chinese_cabbage


In [ ]:
# Visualize the three core CGI features (GC content, obs/exp ratio, length)
# in 3D to check whether species form visually separable clusters
fig = px.scatter_3d(islands, x='gc-content', y='obs/exp', z='length', color='species')
fig.update_traces(marker=dict(size=1))
fig.show()

In [ ]:
NUCLEOTIDE = ['A', 'T', 'G', 'C']
K_MERS = [''.join(n) for n in product(NUCLEOTIDE, repeat=3)]

In [ ]:
def count_3mer(seq, length):
    """
    Count all 3-mer (trinucleotide) occurrences in a sequence and
    normalize by sequence length, so islands of different lengths
    are comparable on the same scale.

    Args:
        seq (str): nucleotide sequence
        length (int): sequence length used for normalization

    Returns:
        dict: {3-mer: normalized frequency} for all 64 possible 3-mers
              (0 if absent)
    """
  counts = Counter([seq[i:i+3] for i in range(len(seq) - 2)])

  return {k: counts.get(k, 0)/length for k in K_MERS}

In [ ]:
# Extract normalized 3-mer composition for every CGI sequence
features = []

for island in islands.itertuples():
  feature = {'id': island.id}
  feature.update(count_3mer(island.seq, island.length))
  features.append(feature)

island_features = pd.DataFrame(features)

In [ ]:
island_features

,id,AAA,AAT,AAG,AAC,ATA,ATT,ATG,ATC,AGA,...,CTG,CTC,CGA,CGT,CGG,CGC,CCA,CCT,CCG,CCC
0,AT1G14150,0.006098,0.018293,0.012195,0.024390,0.012195,0.012195,0.009146,0.030488,0.033537,...,0.021341,0.027439,0.015244,0.006098,0.021341,0.018293,0.024390,0.012195,0.024390,0.003049
1,AT1G14140,0.000000,0.018349,0.013761,0.013761,0.022936,0.004587,0.022936,0.027523,0.022936,...,0.018349,0.036697,0.013761,0.022936,0.013761,0.022936,0.009174,0.032110,0.032110,0.009174
2,AT1G64960,0.000000,0.004739,0.037915,0.000000,0.004739,0.009479,0.023697,0.009479,0.014218,...,0.018957,0.004739,0.014218,0.009479,0.014218,0.004739,0.009479,0.009479,0.014218,0.004739
3,AT1G08270,0.018779,0.018779,0.051643,0.014085,0.009390,0.014085,0.014085,0.018779,0.056338,...,0.014085,0.009390,0.023474,0.009390,0.037559,0.018779,0.014085,0.000000,0.014085,0.014085
4,AT1G08450,0.020690,0.018391,0.025287,0.022989,0.016092,0.013793,0.016092,0.011494,0.022989,...,0.009195,0.013793,0.027586,0.025287,0.022989,0.013793,0.020690,0.013793,0.013793,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
19345,BraA10g033540.4C.1,0.014493,0.000000,0.019324,0.028986,0.004831,0.019324,0.024155,0.004831,0.009662,...,0.009662,0.004831,0.019324,0.009662,0.004831,0.004831,0.004831,0.009662,0.014493,0.000000
19346,BraA10g033560.4C.1,0.033898,0.014528,0.031477,0.019370,0.007264,0.004843,0.021792,0.007264,0.007264,...,0.014528,0.029056,0.007264,0.016949,0.007264,0.009685,0.031477,0.014528,0.007264,0.021792
19347,BraA10g033750.4C.1,0.025037,0.017673,0.023564,0.020619,0.013255,0.010309,0.020619,0.007364,0.014728,...,0.010309,0.014728,0.019146,0.017673,0.036819,0.013255,0.016200,0.008837,0.020619,0.002946
19348,BraA10g033840.4C.1,0.009259,0.006173,0.030864,0.015432,0.006173,0.021605,0.015432,0.006173,0.024691,...,0.015432,0.021605,0.003086,0.024691,0.015432,0.021605,0.018519,0.009259,0.024691,0.012346


In [ ]:
pca = PCA(n_components=2)
island_features_pca = pca.fit_transform(island_features.iloc[:, 1:])

In [ ]:
print(f"PC1 explained variance: {pca.explained_variance_ratio_[0]:.2%}")
print(f"PC2 explained variance: {pca.explained_variance_ratio_[1]:.2%}")
print(f"Total explained variance: {sum(pca.explained_variance_ratio_):.2%}")

PC1 설명력: 17.85%
PC2 설명력: 17.70%
전체 설명력: 35.55%


In [ ]:
kmer_pca = pd.DataFrame(island_features_pca, columns=["PC1", "PC2"])
kmer_pca['species'] = islands['species']
kmer_pca

,PC1,PC2,species
0,0.019261,0.004090,arabidopsis
1,0.026470,-0.018946,arabidopsis
2,-0.015590,0.042600,arabidopsis
3,-0.027086,0.033194,arabidopsis
4,-0.005716,0.009994,arabidopsis
...,...,...,...
19345,-0.020556,0.038797,chinese_cabbage
19346,0.015298,0.001801,chinese_cabbage
19347,-0.020179,-0.001159,chinese_cabbage
19348,-0.000795,0.010668,chinese_cabbage


In [ ]:
fig = px.scatter(kmer_pca, x="PC1", y='PC2', color='species')
fig.show()

In [ ]:
#Elbow method to find optimal component number of pca
#(no clear elbow found in practice; n=11 was chosen as the smallest n
# reaching >=70% cumulative explained variance — see clustering.ipynb)
pca_performance = {'n':[], 'explain':[]}

for n in range(2, 15):
  pca = PCA(n_components=n)
  island_features_pca = pca.fit_transform(island_features.iloc[:, 1:])

  pca_performance['n'].append(n)
  pca_performance['explain'].append(round(sum(pca.explained_variance_ratio_),2))

pca_performance = pd.DataFrame(pca_performance)

In [ ]:
fig = px.line(pca_performance, x='n', y='explain')
fig.show()

In [ ]:
island_features.to_csv('../dataset/processed/3mer_features.csv', index=False)
islands.to_csv('../dataset/processed/cpg_islands.csv', index=False)